# Ensemble Voting Test
This notebook demonstrates soft and hard voting with a random image from the dataset.

In [1]:
import sys
from pathlib import Path

current_dir = Path.cwd()
proj_dir = current_dir.parent.parent.resolve() # get project root directory

print(f"Project directory set to: {proj_dir}")

# Add the project root to the system path to import local modules
sys.path.append(str(proj_dir / 'python'))

Project directory set to: E:\Admin\Documents\dermascan


In [2]:
from dataset_manager import DatasetManager
from voter import EnsembleVoter, EnsembleBaseModel
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np

In [3]:
manager = DatasetManager(data_dir=str(proj_dir / "data"))

# Load metadata (filtering for Melanoma vs Nevi as per trainer restrictions)
manager.load_cleaned_metadata(filter_melanoma_only=True)

In [4]:
# Subclasses for the two types of models (RGB and Filtered)
class ResNetRGBModel(EnsembleBaseModel):
    def load_model(self):
        model = models.resnet18()
        model.fc = nn.Linear(model.fc.in_features, 2)
        model.load_state_dict(torch.load(self.model_path, map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu')))
        model.eval()
        return model

    def load_image(self, image_info):
        # image_info is assumed to be the row of the dataframe
        img_path = image_info['image_path']
        img = Image.open(img_path).convert('RGB')
        
        tensor = transforms.functional.to_tensor(img)
        return tensor.unsqueeze(0) # Add batch dimension as pyTorch models expect (B, C, H, W)

    def predict_proba(self, image_tensor):
        with torch.no_grad():
            outputs = self.model(image_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)
        return probs.squeeze(0).tolist()


class ResNetFilterModel(EnsembleBaseModel):
    def __init__(self, model_id: str, model_path: str, filter_name: str, input_channels: int):
        self.filter_name = filter_name
        self.in_channels = input_channels
        super().__init__(model_id, model_path)

    def load_model(self):
        model = models.resnet18()
        
        # Adapt first conv layer for current filter channels
        old_conv = model.conv1
        model.conv1 = nn.Conv2d(
            self.in_channels, 
            old_conv.out_channels, 
            kernel_size=old_conv.kernel_size, 
            stride=old_conv.stride, 
            padding=old_conv.padding, 
            bias=old_conv.bias
        )
        model.fc = nn.Linear(model.fc.in_features, 2)
        
        model.load_state_dict(torch.load(self.model_path, map_location=torch.device('cuda' if torch.cuda.is_available() else 'cpu')))
        model.eval()
        return model

    def load_image(self, image_info):
        filtered_path = Path(str(proj_dir) + "/data/filtered")
        # Load npy file directly using image_id and the filter_name folder
        filtered_features = np.load(filtered_path / self.filter_name / (image_info['image_id'] + ".npy"))
        
        # Convert to Tensor directly without transforms
        tensor = torch.from_numpy(filtered_features).float()
        
        if len(tensor.shape) == 3:
            # Add batch dimension for multi-channel filters like Gabor, GLCM
            tensor = tensor.unsqueeze(0)
        elif len(tensor.shape) == 2:
            # Add channel dimension and batch dimension for single-channel filters like LBP
            tensor = tensor.unsqueeze(0).unsqueeze(0)
            
        return tensor

    def predict_proba(self, image_tensor):
        with torch.no_grad():
            outputs = self.model(image_tensor)
            probs = torch.nn.functional.softmax(outputs, dim=1)
        return probs.squeeze(0).tolist() # 

specs = [
    ResNetRGBModel(
        model_id="resnet18_rgb", 
        model_path=str(proj_dir / "python/results/models/ResNet_25022026-195039_rgb.pt")
    ),
    ResNetFilterModel(
        model_id="resnet18_gabor", 
        model_path=str(proj_dir / "python/results/models/ResNet_06032026-181320_gabor_filter.pt"),
        filter_name="gabor_filter",
        input_channels=40
    ),
    ResNetFilterModel(
        model_id="resnet18_lbp", 
        model_path=str(proj_dir / "python/results/models/ResNet_15032026-153758_lbp_filter.pt"),
        filter_name="lbp_filter",
        input_channels=1
    ),
]

### Soft Voting

In [5]:
voter = EnsembleVoter(
    model_specs=specs,
    strategy="soft",
    weights={"resnet_rgb": 0.5, "resnet_gabor": 0.5},
    tie_breaker="max_prob",
    label_names=["nv", "mel"],
)

# Grab a random row from the metadata for testing
test_image_row = manager.df.sample(n=1).iloc[0]

print(f"Testing on image: {test_image_row['image_id']} (Real Label: {test_image_row['lesion_idx']})")

result = voter.run(image=test_image_row)
print(f"\nSoft Voting Result: Predicted '{result.label}' with confidence {result.confidence:.4f}")

Testing on image: ISIC_0029004 (Real Label: 0)
Ensemble voting summary
Strategy: soft
Model           | Predicted       | Match? | Confidence | Prob(Final)
------------------------------------------------------------------
resnet18_rgb    | nv              | Yes    | 0.9987     | 0.9987     
resnet18_gabor  | nv              | Yes    | 0.9939     | 0.9939     
resnet18_lbp    | nv              | Yes    | 0.8989     | 0.8989     
------------------------------------------------------------------
Ensemble        | nv              | Yes    | 0.9638     | 0.9638     

Soft Voting Result: Predicted '0' with confidence 0.9638


### Hard Voting

In [6]:
# Hard voting with a tie breaker
voter_hard = EnsembleVoter(
    model_specs=specs,
    strategy="hard",
    tie_breaker="max_prob",
    label_names=["nv", "mel"],
)

result_hard = voter_hard.run(image=test_image_row)
print(f"\nHard Voting Result: Predicted '{result_hard.label}' with confidence {result_hard.confidence:.4f}")

Ensemble voting summary
Strategy: hard
Model           | Predicted       | Match? | Confidence | Prob(Final)
------------------------------------------------------------------
resnet18_rgb    | nv              | Yes    | 0.9987     | 0.9987     
resnet18_gabor  | nv              | Yes    | 0.9939     | 0.9939     
resnet18_lbp    | nv              | Yes    | 0.8989     | 0.8989     
------------------------------------------------------------------
Ensemble        | nv              | Yes    | 1.0000     | 1.0000     

Hard Voting Result: Predicted '0' with confidence 1.0000
